# Page 10 — the search arm: what moving the judge inside the loop bought

Page 10's own header states the thesis: *"Everything else on this page is best-of-N
(page 8) with the judge moved inside the loop: a verifier scores partial lines while they
are still being written (page 9), and the search decides which lines to extend."*

Section 2 demonstrates that on one fixed 29-word tree, where every node is visible and
each algorithm's signature is legible in time. This notebook asks the same question on
the deck's real instrument: pages 6 to 8's 100 MATH-500 items, the same 3B generator, and
**the same judge page 8 used**, moved from scoring finished chains to scoring partial
ones.

The result is one small panel beside Yao et al. in section 3. It does not replace the
tree, which is the page's centrepiece and which no real MATH search could ever render.

## What this can and cannot run, stated first

Of section 2's five strategies, **two survive contact with a real reasoning tree.**

| strategy | on MATH-500 | why |
|---|---|---|
| greedy | yes, already measured (page 6) | one line, no way back |
| beam | **yes, this notebook** | fixed frontier, b lines carried forward |
| best-of-N | yes, already measured (page 8) | the judge, applied after |
| breadth-first | no | the frontier is unbounded once steps are free text |
| depth-first | no | no terminal test short of generating to the end of every branch |
| MCTS | no | needs rollouts to terminal per node, thousands per item |

That is a limit of our hardware and of free-text reasoning steps, not a claim about the
algorithms, and the page says so in those words. The honest pair is **greedy against
beam**: exactly the toy tree's own lesson, carried onto the instrument.

## How to run this

| Mode | What runs | Needs | Time |
|---|---|---|---|
| `REPLAY` | gates 1 and 2: the ladder's lower rungs and the judge, both re-derived | pages 6/7/8's `out/`, ~4 GB GPU for gate 2 | ~10 min |
| `QUICK` | + gate 3, the beam smoke test on 20 items | ~10 GB GPU | +~25 min |
| `FULL` | + the beam on all 100 | a 16 GB GPU | ~2 h |

**Two of the ladder's three rungs cost nothing**: greedy is page 6's, best-of-16 is
page 8's, on these same items. Only the top rung is generated here.

In [ ]:
# ---------------------------------------------------------------------------
# The knobs. Everything else is derived from these.
# ---------------------------------------------------------------------------
RUN_MODE = "REPLAY"      # "REPLAY" (gates 1-2) | "QUICK" (+gate 3) | "FULL" (+the arm)
SEED = 123               # pages 6-8's seed: the item draw must be their draw

PRIMARY = "Qwen/Qwen2.5-3B-Instruct"     # Act 2's shared instrument, pages 6-8's writer
BIG = "Qwen/Qwen2.5-7B-Instruct"         # context only: the target page 11 prices against

# ---- which judge steers the search -------------------------------------------------
# Two, and running both is the point. Page 8 chose its judge deliberately and wrote down
# why: "An OUTCOME reward model keeps page 9's process-supervision story intact; a process
# reward model would win more easily and spend page 9's subject to do it."
#
#   "outcome"  Skywork, page 8's judge, a value pooled at the LAST token of a solution.
#              Holding it fixed isolates timing, but it was trained to score FINISHED
#              work, so asking it to value a prefix asks a question it never learned.
#              Measured here at 40%, below greedy's 43%: the diagnostic rung.
#   "process"  Page 9's judge, trained to score every step. This is the fair test of
#              "the judge inside the loop", and page 10 CONSUMES page 9's conclusion
#              (only process supervision reads steps) rather than re-deriving it.
JUDGE_KIND = "process"          # "process" | "outcome"
OUTCOME_JUDGE = "Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B"
PROCESS_JUDGE = "Qwen/Qwen2.5-Math-PRM-7B"
JUDGE = PROCESS_JUDGE if JUDGE_KIND == "process" else OUTCOME_JUDGE

# None means bf16. Page 9 ran this PRM at 8-bit because 15 GB of weights do not fit
# beside a generator on a 16 GB card, and recorded that as a property of its run. On a
# 40 GB card there is no reason to quantise at all, and gate 2b measures what the choice
# was worth either way rather than assuming it was free.
PRM_QUANT = None                # None (bf16) | "8bit" | "4bit"
STEP_SEP = "<extra_0>"          # the token the process PRM scores at, one per step
MIN_STEP_CHARS = 12             # page 9's, so a chain segmented here breaks as it does there

# What a partial line is worth. The process PRM gives one probability per step, so a
# prefix has a vector and the beam needs a scalar. "last" asks how good the state is NOW,
# which is the value-function reading a search wants; "min" asks whether anything has
# gone wrong yet; "mean" smooths. Recorded because it is a choice, not a default.
VALUE_AT = "last"               # "last" | "min" | "mean"

HARD_LEVELS, HARD_N = (4, 5), 100

# ---- the beam -----------------------------------------------------------------------
BEAM_WIDTH = 4           # lines carried forward on the full arm
EXPAND = 4               # continuations sampled per live line per step

# Gate 3 sweeps these before the arm runs, and the first entry is the control that makes
# the rest interpretable. (1, 1) is no selection at all: one line, chopped at step
# boundaries and resumed, with the judge never choosing anything. If it lands near the
# single-sampled-chain baseline then the chopping machinery is harmless and every number
# above it is about SEARCH. If it craters, the harness is the story and nothing else here
# means what it appears to mean.
SWEEP = [(1, 1), (2, 4), (4, 4)]
MAX_DEPTH = 12           # steps before a line is forced to commit
STEP_MAX_TOKENS = 256    # a step that runs longer than this is cut and scored anyway
TOTAL_CAP = 2048         # the SAME per-line budget as pages 6-8, so the axis is theirs
STEP_STOP = "\n\n"       # page 9's first cascade rule: a step ends at a blank line.
                         # Named here because it is a claim about where a step ends, and
                         # the judge was trained on responses split exactly this way.
T_STEP = 0.6             # step sampling temperature; page 8's best-of-N pool used 0.60
TOP_P = 0.95
ANSWER_ALLOWANCE = 32    # page 7's forced-commit allowance for a line that never boxed

FORCE_STEM = "\n\nFinal answer: \\boxed{"
RM_MAX_TOKENS = 8192

# ---- gate 2: does the rebuilt judge still score like page 8's? ----------------------
GATE2_SAMPLE = 60        # of page 8's own scored candidates, re-scored here
GATE2_CHAINS = 120       # process gate: chains re-scored (one 7B forward each). None = all
GATE2_MAX_DRIFT = 1e-3   # same weights, same head, same reading position -> same number

# ---- gate 3, pre-registered before any beam output was seen ------------------------
# Beam has to clear the rung below it. Best-of-16 with this judge scores 60 on these
# items; a beam that cannot reach the region of that number on a smoke sample is not
# going to on the full run, and the honest move is to report that rather than spend it.
GATE3_ITEMS = 20
GATE3_MIN = 0.45

FLOPS = lambda params, tokens: 2.0 * params * tokens

## Running this on a rented card

The arm wants both models resident and unquantised, which is ~21 GB: comfortable on an
A100, impossible on 16 GB. **Pick the runtime before anything else** (Runtime > Change runtime type). The arm holds
a 3B generator beside a 7B PRM, so the card decides the precision, not the other way
round:

| card | VRAM | PRM precision |
|---|---|---|
| A100 | 40 GB | `PRM_QUANT = None` (bf16), nothing quantised |
| **L4** | 24 GB | `PRM_QUANT = "8bit"` — measured AUC 0.8905 against page 9's 0.8922 |
| T4 | 16 GB | not enough at any precision worth using |

The cell below prints what fits and refuses a combination that does not, rather than
letting you discover it twenty minutes into a 21 GB download.

Then upload `page-10-search-arm.zip` beside this notebook and run the cell. It unpacks
the bundle and installs what Colab is missing. It is a no-op wherever imports already work.

In [ ]:
# Colab bootstrap. Does nothing when harness.py is already importable.
import importlib.util, subprocess, sys, zipfile
from pathlib import Path

if importlib.util.find_spec("harness") is None:
    bundle = next(iter(Path(".").glob("page-10-search-arm*.zip")), None)
    if bundle is None:
        try:
            from google.colab import files
            print("upload page-10-search-arm.zip")
            bundle = Path(next(iter(files.upload())))
        except Exception as exc:
            raise SystemExit("no bundle found and not on Colab: place "
                             "page-10-search-arm.zip beside this notebook") from exc
    with zipfile.ZipFile(bundle) as z:
        z.extractall(".")
    # Everything EXCEPT torch. Colab ships a torch built against its own CUDA, and
    # letting pip honour requirements.txt's pin would reinstall it: slow at best, and at
    # worst it leaves a torch that cannot see the GPU.
    reqs = [l.split("#")[0].strip()
            for l in Path("requirements.txt").read_text().splitlines()]
    reqs = [r for r in reqs if r and not r.lower().startswith("torch")]
    print("installing:", ", ".join(reqs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *reqs])
    print("\nbundle unpacked. RESTART THE RUNTIME now "
          "(Runtime > Restart session), then run from the top.")
else:
    print("harness importable; bootstrap skipped")

try:
    import torch
    if torch.cuda.is_available():
        gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        name = torch.cuda.get_device_name(0)
        print(f"\ngpu: {name}, {gb:.0f} GB")
        # Measured on this repo's 16 GB card, not estimated: the 3B generator is
        # 6.2 GB bf16, the 7B PRM is 15.2 / 7.9 / 4.9 GB at bf16 / 8bit / 4bit, and a
        # 4x4 beam step adds ~0.7 GB on top. 2 GB of slack for the driver and the
        # allocator's fragmentation.
        NEED = {"bf16": 6.2 + 15.2, "8bit": 6.2 + 7.9, "4bit": 6.2 + 4.9}
        for q in ("bf16", "8bit", "4bit"):
            fits = gb >= NEED[q] + 0.7 + 2.0
            print(f"  PRM {q:<5} needs ~{NEED[q] + 0.7 + 2.0:4.1f} GB  "
                  f"{'fits' if fits else 'does NOT fit'}")
        want = "bf16" if PRM_QUANT is None else PRM_QUANT
        ok = [q for q in ("bf16", "8bit", "4bit") if gb >= NEED[q] + 2.7]
        assert ok, (
            f"{name} has {gb:.0f} GB and cannot hold the 3B generator beside the 7B PRM "
            "at any precision. An L4 (24 GB) or better is the floor.")
        assert want in ok, (
            f"{name} has {gb:.0f} GB, which is not enough for PRM_QUANT={want!r}. "
            f"Set PRM_QUANT = {ok[0]!r}. Gate 2 re-measures the judge's AUC at whatever "
            "precision loads, so the cost of that choice is reported rather than assumed "
            "(8-bit measured 0.8905 against page 9's 0.8922).")
        if want != ok[0]:
            print(f"  -> running at {want}; {ok[0]} would also fit")
    else:
        print("\nno GPU visible: Runtime > Change runtime type > A100 GPU")
except ImportError:
    pass

In [ ]:
import json
import math
import random
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import torch

import harness
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
    load_math500_test, render_prompt, extract_final_candidate, grade_answer,
)

OUT_DIR = Path("out/page_10"); OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = Path("data"); DATA_DIR.mkdir(parents=True, exist_ok=True)
TRACES_PATH = OUT_DIR / "traces.jsonl"
DERIVED_PATH = OUT_DIR / "derived.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
UP = {n: Path(f"out/page_{n:02d}") for n in (6, 7, 8, 9)}
GATES = {}


def read_traces(path):
    latest = {}
    for line in Path(path).open(encoding="utf-8"):
        if not line.strip():
            continue
        r = json.loads(line)
        latest[(r["run_id"], r["model_id"], r.get("config_id", r.get("budget")),
                r["problem_id"], r.get("sample_index", 0))] = r
    return list(latest.values())


mean, median = harness.mean, harness.median
device = get_device()
DTYPE = harness.pick_dtype(device)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", device)
print("harness", harness.fingerprint()["version"], harness.fingerprint()["sha256"][:12])
print("mode", RUN_MODE)

## 1. The items, and the two rungs already paid for

The band is pages 6 to 8's exact draw. Greedy and best-of-16 are read from their runs
rather than regenerated: the ladder's novelty is its top rung, and re-measuring the
bottom two would only introduce a chance to disagree with the deck about its own numbers.

In [ ]:
math500 = load_math500_test(local_path=DATA_DIR / "math500_test.json")
band = sorted((r for r in math500 if int(r["level"]) in HARD_LEVELS),
              key=lambda r: r["unique_id"])
random.Random(SEED).shuffle(band)
items = band[:HARD_N]
ITEM_HASH = harness.sha256_of(r["unique_id"] for r in items)
print(f"band L{HARD_LEVELS}  n={len(items)}  sha256 {ITEM_HASH[:12]}")

p7m = UP[7] / "manifest.json"
if p7m.exists():
    want = json.loads(p7m.read_text())["items"]["item_set_sha256"]
    assert ITEM_HASH == want, ("this is NOT pages 6-8's item draw; every rung below would "
                               "describe a different population. Match SEED and HARD_N.")
    print("-> verified against page 7's manifest")

RUNGS = {}
if (UP[6] / "derived.json").exists():
    d6 = json.loads((UP[6] / "derived.json").read_text())
    c = next(x for x in d6["figure_2"]["curves"] if x["model_id"] == PRIMARY)
    pt = next(p for p in c["points"] if p["budget"] == max(d6["budgets"]))
    RUNGS["greedy"] = dict(label="greedy, one line, no way back", accuracy=pt["accuracy"],
                           mean_tokens=pt["mean_tokens"], params=c["params"],
                           source="page 6", judge_calls=0)
    PARAMS = c["params"]
if (UP[7] / "derived.json").exists():
    d7 = json.loads((UP[7] / "derived.json").read_text())
    h = d7["handoff"]
    RUNGS["vote"] = dict(label=f"majority vote@{h['n_chains']}, no judge",
                         accuracy=h["plurality_accuracy"], source="page 7",
                         coverage=h["coverage"], judge_calls=0)
if (UP[8] / "derived.json").exists():
    d8 = json.loads((UP[8] / "derived.json").read_text())
    b8 = d8["gate"]["best_of_n"]
    RUNGS["best_of_n"] = dict(label=f"{b8['selector']}@{b8['n']}, judge after",
                              accuracy=b8["accuracy"], n=b8["n"],
                              config_id=b8["config_id"], verifier=b8["verifier"],
                              source="page 8", judge_calls=b8["n"])
    assert b8["verifier"] == "skywork-prm", (
        "page 8's headline selector is not the skywork PRM, so the judge this notebook "
        "holds fixed is not the one that produced the rung below it")

for k, v in RUNGS.items():
    print(f"  {k:<11} {v['accuracy']:.0%}  ({v['source']})  {v['label']}")

## 2. Gate 1 — the vote and coverage rules, ported and re-derived

The same drift trap every measured page in this deck carries: pages 7's aggregates,
recomputed here from its own traces, must match its shipped `derived.json` exactly.

In [ ]:
NO_ANSWER = None


def vote_key(text):
    cand = extract_final_candidate(text, fallback="number_only")
    return cand if cand and cand.strip() else NO_ANSWER


_EQ = {}


def equivalent(a, b):
    if a == b:
        return True
    k = (a, b) if a <= b else (b, a)
    if k not in _EQ:
        try:
            _EQ[k] = bool(grade_answer(a, b)) or bool(grade_answer(b, a))
        except Exception:
            _EQ[k] = False
    return _EQ[k]


def cluster(keys):
    labels, order = [], []
    for k in keys:
        if k is NO_ANSWER:
            labels.append(NO_ANSWER); continue
        hit = next((l for l in order if equivalent(l, k)), None)
        if hit is None:
            order.append(k); hit = k
        labels.append(hit)
    return labels, order


def tally(records):
    gt = records[0]["ground_truth"]
    labels, _ = cluster([vote_key(r["trace"]) for r in records])
    counts = Counter([l for l in labels if l is not NO_ANSWER])
    ranked = counts.most_common()
    top = ranked[0][1] if ranked else 0
    winners = [a for a, c in ranked if c == top]
    tie = len(winners) > 1
    win = winners[0] if (winners and not tie) else None
    return dict(n_chains=len(records), tie=tie,
                plurality_correct=bool(win is not None and grade_answer(win, gt)),
                any_correct=any(r["correct"] for r in records),
                mean_chain_correct=mean([r["correct"] for r in records]),
                truncated_rate=mean([r["truncated"] for r in records]),
                tokens_generated=sum(r["tokens_generated"] for r in records))


GATE1 = dict(reproduced=False, skipped=True)
if (UP[7] / "traces.jsonl").exists():
    raw = read_traces(UP[7] / "traces.jsonl")
    d7 = json.loads((UP[7] / "derived.json").read_text())
    want_ids = {r["unique_id"] for r in items}
    run = Counter(r["run_id"] for r in raw).most_common(1)[0][0]
    raw = [r for r in raw if r["run_id"] == run and r["model_id"] == PRIMARY
           and r["problem_id"] in want_ids]
    pool = {}
    for r in raw:
        if r["config_id"] == d7["pool"]["config_id"]:
            pool.setdefault(r["problem_id"], []).append(r)
    for pid in pool:
        pool[pid].sort(key=lambda r: r["sample_index"])
    ship = d7["handoff"]
    rows = [tally(pool[p["unique_id"]][:ship["n_chains"]])
            for p in items if p["unique_id"] in pool]
    got = dict(plurality_accuracy=mean([r["plurality_correct"] for r in rows]),
               coverage=mean([r["any_correct"] for r in rows]),
               single_chain_accuracy=mean([r["mean_chain_correct"] for r in rows]))
    drift = {k: abs(got[k] - ship[k]) for k in got}
    for k, dv in drift.items():
        print(f"  {k:<24}{ship[k]:>8.4f} ->{got[k]:>8.4f}   |d|={dv:.2e}")
    GATE1 = dict(reproduced=bool(max(drift.values()) < 1e-12), skipped=False,
                 max_abs_drift=max(drift.values()))
    assert GATE1["reproduced"], "the ported selectors drifted from page 7"
else:
    print("page 7 traces not on disk; gate 1 cannot run")

GATES["gate1"] = GATE1
print(f"gate 1: {'PASSED' if GATE1['reproduced'] else 'NOT RUN' if GATE1['skipped'] else 'FAILED'}")

## 3. The two judges

**The outcome judge** is page 8's Skywork RM, rebuilt on the maintained Qwen2 body
because the shipped modelling code returns NaN under transformers 5.x. Its head pools one
value at the last token, which is what makes page 8's best-of-N outcome-supervised.

**The process judge** is page 9's `Qwen2.5-Math-PRM-7B`, rebuilt the same way and for the
same reason: its bundled code returns *finite but wrong* hidden states, scoring every step
near 0.25 and separating a known-false step from a true one by 0.002. That reads as "this
PRM is useless on our chains" and actually means "we computed it wrong." Page 9's Paris
preflight below is what caught it, and it is carried here unchanged.

A prefix has one probability per step, so `VALUE_AT` turns the vector into the scalar a
beam needs. That is a choice and it is recorded.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

judge_body = judge_head = judge_tok = None
JUDGE_META = {}


class ValueHeadRM:
    """Page 8's outcome RM: stock Qwen2 body plus one hidden_size -> 1 head, read at the
    last token. Rebuilt rather than loaded because the shipped modelling code returns NaN
    under transformers 5.x, which would score every line identically."""

    def __init__(self, model_id, quant=None):
        from transformers import BitsAndBytesConfig, Qwen2Config, Qwen2Model
        from huggingface_hub import snapshot_download

        path = Path(snapshot_download(model_id))
        sd = torch.load(path / "pytorch_model.bin", map_location="cpu",
                        weights_only=True, mmap=True)
        body = {k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}
        head = {k: v for k, v in sd.items() if k.startswith("v_head.")}
        assert head, f"{model_id} carries no v_head.* tensors; wrong loader for it"
        cfg = Qwen2Config.from_pretrained(path)
        self.body = Qwen2Model(cfg)
        missing, unexpected = self.body.load_state_dict(body, strict=False)
        assert not missing and not unexpected, (
            f"{model_id} does not map cleanly onto transformers' Qwen2: "
            f"missing={list(missing)[:5]} unexpected={list(unexpected)[:5]}")
        self.body = self.body.to(dtype=harness.pick_dtype(device), device=device).eval()
        self.w = head["v_head.summary.weight"].to(device, torch.float32).flatten()
        self.b = head["v_head.summary.bias"].to(device, torch.float32)
        self.params = sum(p.numel() for p in self.body.parameters())

    @torch.inference_mode()
    def value(self, enc):
        h = self.body(**enc, use_cache=False).last_hidden_state[0, -1].float()
        return float(h @ self.w + self.b)


def load_process_prm(model_id, quant):
    """Page 9's loader: a stock Qwen2 body plus the checkpoint's own 2-class head."""
    from safetensors import safe_open
    from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen2Model

    path = Path(harness.fetch(model_id))
    tok = AutoTokenizer.from_pretrained(path)
    if quant in ("8bit", "4bit"):
        # The head is skipped: its output dimension is 2 and bitsandbytes' int8 matmul
        # fails on a matrix that narrow. It is 26 MB, so full precision there is free.
        cfg = (BitsAndBytesConfig(load_in_8bit=True, llm_int8_skip_modules=["score"])
               if quant == "8bit" else
               BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.bfloat16,
                                  llm_int8_skip_modules=["score"]))
        body = Qwen2Model.from_pretrained(path, quantization_config=cfg).eval()
    else:
        body = Qwen2Model.from_pretrained(
            path, dtype=harness.pick_dtype(device)).to(device).eval()

    index = path / "model.safetensors.index.json"
    keys = ("score.0.weight", "score.0.bias", "score.2.weight", "score.2.bias")
    where = (json.loads(index.read_text())["weight_map"] if index.exists()
             else {k: "model.safetensors" for k in keys})
    w = {}
    for k in keys:
        with safe_open(path / where[k], framework="pt") as sf:
            w[k] = sf.get_tensor(k)
    h = body.config.hidden_size
    head = torch.nn.Sequential(torch.nn.Linear(h, h), torch.nn.ReLU(),
                               torch.nn.Linear(h, 2))
    head[0].weight.data, head[0].bias.data = w[keys[0]].float(), w[keys[1]].float()
    head[2].weight.data, head[2].bias.data = w[keys[2]].float(), w[keys[3]].float()
    return body, head.to(next(body.parameters()).device).float().eval(), tok


def load_judge():
    global judge_body, judge_head, judge_tok, JUDGE_META
    if judge_tok is not None:
        return
    if JUDGE_KIND == "process":
        judge_body, judge_head, judge_tok = load_process_prm(JUDGE, PRM_QUANT)
        params = sum(p.numel() for p in judge_body.parameters())
    else:
        from transformers import AutoTokenizer
        judge_body = ValueHeadRM(JUDGE, PRM_QUANT)
        judge_tok = AutoTokenizer.from_pretrained(JUDGE)
        params = judge_body.params
    JUDGE_META = dict(id=JUDGE, kind=JUDGE_KIND, quant=PRM_QUANT or "bf16",
                      params=params, value_at=VALUE_AT, calls=0, tokens=0)
    print(f"judge ready: {JUDGE} ({JUDGE_KIND}, {PRM_QUANT or 'bf16'}, {params:,} params)")


@torch.inference_mode()
def prm_step_probs(question, text):
    """One probability per step for a partial or finished line. Page 9's formatting
    exactly: the judge is calibrated to it and reformatting changes what is measured."""
    steps = [t.strip() for t in re.split(r"\n\s*\n", harness.strip_empty_think(text))
             if len(t.strip()) >= MIN_STEP_CHARS]
    if not steps:
        return []
    body_text = STEP_SEP.join(steps) + STEP_SEP
    chat = judge_tok.apply_chat_template(
        [{"role": "system", "content": "Please reason step by step."},
         {"role": "user", "content": question},
         {"role": "assistant", "content": body_text}],
        tokenize=False, add_generation_prompt=False)
    enc = judge_tok(chat, return_tensors="pt", truncation=True,
                    max_length=RM_MAX_TOKENS).to(next(judge_body.parameters()).device)
    hidden = judge_body(**enc, use_cache=False).last_hidden_state
    probs = torch.softmax(judge_head(hidden.float()), dim=-1)
    sep_id = judge_tok.encode(STEP_SEP, add_special_tokens=False)[0]
    JUDGE_META["calls"] += 1
    JUDGE_META["tokens"] += int(enc["input_ids"].shape[1])
    return probs[enc["input_ids"] == sep_id][:, 1].tolist()


@torch.inference_mode()
def outcome_value(question, text):
    chat = judge_tok.apply_chat_template(
        [{"role": "user", "content": question}, {"role": "assistant", "content": text}],
        tokenize=False)
    enc = judge_tok(chat, return_tensors="pt", add_special_tokens=False,
                    truncation=True, max_length=RM_MAX_TOKENS).to(device)
    v = judge_body.value(enc)
    assert v == v, "the reward model returned NaN: a modelling-code version mismatch"
    JUDGE_META["calls"] += 1
    JUDGE_META["tokens"] += int(enc["input_ids"].shape[1])
    return v


def judge_value(question, text):
    """The scalar the beam ranks on."""
    if JUDGE_KIND == "outcome":
        return outcome_value(question, text)
    ps = prm_step_probs(question, text)
    if not ps:
        return 0.0
    return {"last": ps[-1], "min": min(ps), "mean": sum(ps) / len(ps)}[VALUE_AT]


@torch.inference_mode()
def process_body_is_sane():
    """Page 9's check, and the one that caught the real bug. A broken forward pass does
    not announce itself: it returns finite numbers of the right shape and only the
    CONTENT is wrong. So ask it something a 7B maths model cannot get wrong. The
    checkpoint ships an lm_head its reward class never uses, which is exactly the tool."""
    from safetensors import safe_open
    path = Path(harness.fetch(JUDGE))
    index = path / "model.safetensors.index.json"
    where = (json.loads(index.read_text())["weight_map"]["lm_head.weight"]
             if index.exists() else "model.safetensors")
    with safe_open(path / where, framework="pt") as sf:
        w = sf.get_tensor("lm_head.weight")
    dev = next(judge_body.parameters()).device
    w = w.to(torch.float16).to(dev)
    enc = judge_tok("The capital of France is", return_tensors="pt").to(dev)
    hidden = judge_body(**enc, use_cache=False).last_hidden_state
    top = torch.topk((hidden[0, -1].to(torch.float16) @ w.T).float(), 5).indices.tolist()
    words = [judge_tok.decode([i]) for i in top]
    del w
    return any("paris" in x.strip().lower() for x in words), words


print("judge loaders ready (nothing loaded until gate 2 asks)")

## 4. Gate 2 — the judge is the one the deck already measured

The beam steers with this judge, so if our copy of it disagrees with the run that
established it, the ladder compares two judges rather than two timings. The check differs
by kind because what "the same judge" means differs by kind.

**Outcome (2a):** page 8 shipped per-candidate scores, so agreement is exact.

**Process (2b):** page 9 shipped 1,092 checker-labelled steps and reported **AUC 0.892**
against them, with a build-blocking floor of 0.65. We re-score those same steps at
whatever precision loaded and require the floor. That turns the precision question into a
measurement instead of a hope: if bf16 beats page 9's 8-bit number, that is the
quantisation cost page 9 could only name.

In [ ]:
GATE2 = dict(skipped=True)


def auc(scores, labels):
    """Rank AUC, ties averaged. Page 9's quantity."""
    pairs = sorted(zip(scores, labels))
    ranks, i = {}, 0
    while i < len(pairs):
        j = i
        while j + 1 < len(pairs) and pairs[j + 1][0] == pairs[i][0]:
            j += 1
        r = (i + j) / 2 + 1
        for k in range(i, j + 1):
            ranks[k] = r
        i = j + 1
    pos = [ranks[k] for k, (_, l) in enumerate(pairs) if l == 1]
    n_pos, n_neg = len(pos), len(pairs) - len(pos)
    if not n_pos or not n_neg:
        return float("nan")
    return (sum(pos) - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


if JUDGE_KIND == "outcome" and (UP[8] / "scores.jsonl").exists():
    load_judge()
    scores = [json.loads(l) for l in (UP[8] / "scores.jsonl").open() if l.strip()]
    sky = [x for x in scores if x["scorer"] == "skywork-prm"]
    p8 = {(r["config_id"], r["problem_id"], str(r["sample_index"])): r
          for r in read_traces(UP[8] / "traces.jsonl")}
    sample = random.Random(SEED).sample(sky, min(GATE2_SAMPLE, len(sky)))
    diffs = []
    for x in harness.progress(sample, desc="re-scoring page 8", unit="cand"):
        rec = p8.get((x["config_id"], x["problem_id"], str(x["sample_index"])))
        if rec:
            diffs.append(abs(outcome_value(rec["question"], rec["trace"])
                             - float(x["score"])))
    worst = max(diffs) if diffs else float("inf")
    GATE2 = dict(skipped=False, kind="outcome", n=len(diffs), max_abs_drift=worst,
                 passed=bool(diffs and worst < GATE2_MAX_DRIFT))
    print(f"\n{len(diffs)} of page 8's candidates re-scored")
    print(f"  max |ours - page 8|  {worst:.2e}   (gate: < {GATE2_MAX_DRIFT:.0e})")

elif JUDGE_KIND == "process" and (UP[9] / "traces.jsonl").exists():
    load_judge()
    ok, words = process_body_is_sane()
    print(f"body sanity: 'The capital of France is' -> {words}")
    assert ok, ("the PRM body does not know that Paris is in France, which is what a "
                "silently-wrong forward pass looks like. Do not score with it.")

    p9 = read_traces(UP[9] / "traces.jsonl")
    d9 = json.loads((UP[9] / "derived.json").read_text())
    ship = d9["prm_reliability"]
    labelled = [(r, s) for r in p9 for s in r.get("steps", [])
                if s.get("check") in ("true", "false")]
    print(f"\n{len(labelled)} checker-labelled steps on disk "
          f"(page 9 compared {ship['n_steps_compared']})")

    # Re-score each chain once and take the step probabilities in order, which is how
    # page 9 produced its own: scoring steps individually would change the context each
    # one is judged in.
    by_chain, ours, theirs, labels = {}, [], [], []
    for r, st in labelled:
        by_chain.setdefault(id(r), (r, []))[1].append(st)
    chains = list(by_chain.values())
    if GATE2_CHAINS and len(chains) > GATE2_CHAINS:
        chains = random.Random(SEED).sample(chains, GATE2_CHAINS)
        print(f"capped at {GATE2_CHAINS} chains (7B forward per chain is the cost)")
    for r, sts in harness.progress(chains, desc="re-scoring page 9", unit="chain"):
        ps = prm_step_probs(r["question"], r["trace"])
        seq = [s for s in r["steps"] if len(s["text"].strip()) >= MIN_STEP_CHARS]
        for st in sts:
            try:
                i = seq.index(st)
            except ValueError:
                continue
            if i < len(ps):
                ours.append(ps[i]); theirs.append(float(st["prm"]))
                labels.append(1 if st["check"] == "false" else 0)

    # label 1 = the step is FALSE, and a LOW prm probability should flag it, so the
    # discriminating score is the negated probability.
    got = auc([-x for x in ours], labels) if ours else float("nan")
    GATE2 = dict(skipped=False, kind="process", n=len(ours), auc=got,
                 page_9_auc=ship["auc"], floor=ship["gate"]["min_auc"],
                 quant=PRM_QUANT or "bf16", body_sane=ok,
                 mean_abs_drift_vs_page_9=(sum(abs(a - b) for a, b in zip(ours, theirs))
                                           / len(ours)) if ours else None,
                 passed=bool(ours and got >= ship["gate"]["min_auc"]))
    print(f"\n  steps re-scored      {len(ours)}")
    print(f"  AUC at {PRM_QUANT or 'bf16':<6}       {got:.4f}")
    print(f"  page 9 at 8-bit      {ship['auc']:.4f}   (floor {ship['gate']['min_auc']})")
    print(f"  mean |ours - page 9| {GATE2['mean_abs_drift_vs_page_9']:.4f}"
          if GATE2["mean_abs_drift_vs_page_9"] is not None else "")
else:
    print(f"the run needed for gate 2 ({JUDGE_KIND}) is not on disk")

if not GATE2.get("skipped"):
    print(f"gate 2: {'PASSED' if GATE2['passed'] else 'FAILED'}")
    assert GATE2["passed"], (
        "the judge this notebook would steer with is not the one the deck measured; "
        "fix it before generating")

GATES["gate2"] = GATE2

## 5. The beam

At each step every live line is extended `EXPAND` ways, each continuation stopping at a
step boundary; the judge scores each extended prefix; the best `BEAM_WIDTH` survive. A
line ends when it boxes an answer, emits end-of-turn, or exhausts the same 2,048-token
budget pages 6 to 8 gave a single chain.

**The line that is graded is the highest-valued finished line**, which is the search's
own choice rather than a vote over its output. A beam that voted would be page 7 wearing
a different hat.

In [ ]:
gen_model = gen_tok = None
EOS_IDS = set()
GEN_TOKENS = 0


def ensure_weights(model_id):
    from huggingface_hub import snapshot_download
    try:
        snap = Path(snapshot_download(model_id, local_files_only=True))
        if any(snap.glob("*.safetensors")) or any(snap.glob("*.bin")):
            return
        print(f"      cache holds {model_id} without weights; fetching them")
    except Exception:
        pass
    harness.fetch(model_id)


def load_generator(model_id):
    global gen_model, gen_tok, EOS_IDS
    ensure_weights(model_id)
    loaded = harness.load_model(model_id, device=device, dtype=DTYPE)
    gen_model, gen_tok = loaded["model"], loaded["tokenizer"]
    EOS_IDS = set(harness.end_of_turn_ids(gen_tok))
    return loaded


def prompt_ids_for(question):
    chat = gen_tok.apply_chat_template(
        [{"role": "user", "content": render_prompt(question)}],
        tokenize=False, add_generation_prompt=True)
    return gen_tok(chat, add_special_tokens=False).input_ids


@torch.inference_mode()
def extend(prompt_ids, prefix_ids, k, seed):
    """k sampled continuations of one prefix, each cut at the first step boundary."""
    global GEN_TOKENS
    inp = torch.tensor([prompt_ids + prefix_ids], device=gen_model.device)
    torch.manual_seed(seed)
    out = gen_model.generate(
        input_ids=inp, attention_mask=torch.ones_like(inp),
        max_new_tokens=STEP_MAX_TOKENS, do_sample=True, temperature=T_STEP,
        top_p=TOP_P, top_k=None, num_return_sequences=k,
        stop_strings=[STEP_STOP], tokenizer=gen_tok,
        pad_token_id=gen_tok.pad_token_id if gen_tok.pad_token_id is not None
        else gen_tok.eos_token_id)
    steps = []
    for row in out:
        new = row[inp.shape[1]:].tolist()
        # A sequence that hits the stop string early is PADDED to the length of the
        # longest one in its batch, and Qwen's pad token IS its eos token. So "does this
        # contain an eos id" answers "was this padded", not "did the model stop". Truncate
        # at the FIRST eos instead, which keeps the position information that distinguishes
        # the two. (Filtering eos out anywhere, which an earlier version did, marked almost
        # every step terminal and produced 27-token "finished" lines with no answer in them.)
        first_eos = next((i for i, t in enumerate(new) if t in EOS_IDS), None)
        if first_eos is not None:
            new = new[:first_eos]
        GEN_TOKENS += len(new)
        text = gen_tok.decode(new, skip_special_tokens=True)
        # Generation halts just AFTER emitting the stop string, so cut there and keep the
        # boundary itself: the judge was trained on responses that carry it.
        cut = text.find(STEP_STOP)
        hit_stop = cut != -1
        if hit_stop:
            text = text[:cut + len(STEP_STOP)]
            new = gen_tok(text, add_special_tokens=False).input_ids
        # Terminal only when the model ended its turn at a point it chose: an eos with no
        # step boundary before it. Hitting STEP_MAX_TOKENS is neither, and the line lives on.
        steps.append(dict(ids=new, text=text,
                          stopped_eos=bool(first_eos is not None and not hit_stop)))
    return steps


def is_terminal(text, stopped_eos):
    return stopped_eos or "\\boxed{" in text


def config_for(width, expand):
    return f"beam{width}x{expand}-{JUDGE_KIND}-c{TOTAL_CAP}"


@torch.inference_mode()
def beam_search(row, width=None, expand=None):
    """One item. Returns the graded line and the bookkeeping the page needs."""
    width = BEAM_WIDTH if width is None else width
    expand = EXPAND if expand is None else expand
    q = row["problem"]
    pids = prompt_ids_for(q)
    live = [dict(ids=[], text="", value=0.0, depth=0)]
    finished = []
    expansions = 0

    for depth in range(MAX_DEPTH):
        if not live:
            break
        pool = []
        for bi, beam in enumerate(live):
            if len(beam["ids"]) >= TOTAL_CAP:
                continue
            seed = int(harness.sha256_of([str(SEED), row["unique_id"], str(depth),
                                          str(bi), f"{width}x{expand}"])[:15], 16)
            for st in extend(pids, beam["ids"], expand, seed):
                expansions += 1
                text = beam["text"] + st["text"]
                ids = beam["ids"] + st["ids"]
                cand = dict(ids=ids, text=text, depth=depth + 1,
                            value=judge_value(q, text))
                (finished if is_terminal(text, st["stopped_eos"])
                 or len(ids) >= TOTAL_CAP else pool).append(cand)
        pool.sort(key=lambda c: -c["value"])
        live = pool[:width]

    # Nothing boxed: force the best live line to commit, page 7's rule.
    forced = False
    if not finished:
        forced = True
        best = max(live, key=lambda c: c["value"]) if live else dict(ids=[], text="")
        sfx = gen_tok(FORCE_STEM, add_special_tokens=False).input_ids
        inp = torch.tensor([pids + best["ids"] + sfx], device=gen_model.device)
        ans = gen_model.generate(input_ids=inp, attention_mask=torch.ones_like(inp),
                                 max_new_tokens=ANSWER_ALLOWANCE, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=gen_tok.pad_token_id
                                 if gen_tok.pad_token_id is not None else gen_tok.eos_token_id)
        aid = [t for t in ans[0][inp.shape[1]:].tolist() if t not in EOS_IDS]
        text = best["text"] + FORCE_STEM + gen_tok.decode(aid, skip_special_tokens=True)
        finished = [dict(ids=best["ids"] + sfx + aid, text=text,
                         depth=best.get("depth", 0), value=best.get("value", 0.0))]

    # Grade EVERY finished line, not only the one the judge picked. That is the deck's
    # own lens applied to search: if the answer is among the lines the beam produced but
    # the judge did not pick it, that is a selection failure (Stroebl); if it is in none
    # of them, the search never reached it. One number cannot tell those apart.
    def graded_ok(text):
        k = vote_key(text)
        return bool(k is not NO_ANSWER and grade_answer(k, row["answer"]))

    beam_coverage = any(graded_ok(c["text"]) for c in finished)

    win = max(finished, key=lambda c: c["value"])
    # A line can end its turn without ever boxing anything. Grading that as a miss would
    # score the harness rather than the search, so it goes through page 7's forced-commit
    # rule exactly as a capped chain does.
    if vote_key(win["text"]) is NO_ANSWER:
        forced = True
        sfx = gen_tok(FORCE_STEM, add_special_tokens=False).input_ids
        finp = torch.tensor([pids + win["ids"] + sfx], device=gen_model.device)
        ans = gen_model.generate(input_ids=finp, attention_mask=torch.ones_like(finp),
                                 max_new_tokens=ANSWER_ALLOWANCE, do_sample=False,
                                 temperature=None, top_p=None, top_k=None,
                                 pad_token_id=gen_tok.pad_token_id
                                 if gen_tok.pad_token_id is not None else gen_tok.eos_token_id)
        aid = ans[0][finp.shape[1]:].tolist()
        cut = next((i for i, t in enumerate(aid) if t in EOS_IDS), None)
        aid = aid[:cut] if cut is not None else aid
        win = dict(ids=win["ids"] + sfx + aid, value=win["value"], depth=win["depth"],
                   text=win["text"] + FORCE_STEM
                        + gen_tok.decode(aid, skip_special_tokens=True))
    key = vote_key(win["text"])
    return dict(beam_coverage=beam_coverage, width=width, expand=expand,
                problem_id=row["unique_id"], level=str(row["level"]), question=q,
                ground_truth=row["answer"], trace=win["text"],
                tokens_generated=len(win["ids"]), depth=win["depth"],
                value=win["value"], n_finished=len(finished), expansions=expansions,
                forced=forced, truncated=bool(len(win["ids"]) >= TOTAL_CAP),
                extracted_answer=key,
                correct=bool(key is not NO_ANSWER and grade_answer(key, row["answer"])))


# Defined here rather than inside a run branch: section 8 reads it to tell one beam's
# traces from another's. The judge kind is IN the id, so an outcome run and a process run
# share a trace bank without silently pooling into one mislabelled rung.
CONFIG_ID = config_for(BEAM_WIDTH, EXPAND)

print(f"beam ready: width {BEAM_WIDTH}, expand {EXPAND}, depth <= {MAX_DEPTH}, "
      f"line budget {TOTAL_CAP}, config {CONFIG_ID}")

## 6. Gate 3 — the beam smoke test

Pre-registered: 20 items. Best-of-16 with this same judge scores 60 on these items, and a
beam that cannot reach the region of that on a smoke sample is not going to on the full
run. Below the floor, the arm does not run and the page reports the negative as a limit
of our harness rather than of search.

In [ ]:
GATE3 = dict(skipped=True)

if RUN_MODE in ("QUICK", "FULL"):
    load_generator(PRIMARY)
    load_judge()
    RUN_ID = (f"p10-{RUN_MODE.lower()}-{ITEM_HASH[:8]}-b{BEAM_WIDTH}k{EXPAND}"
              f"-d{MAX_DEPTH}-cap{TOTAL_CAP}-t{T_STEP}-{JUDGE_KIND}"
              f"-{PRM_QUANT or 'bf16'}-{device.type}")
    STORE = harness.TraceStore(TRACES_PATH, RUN_ID,
                               ("model_id", "config_id", "problem_id", "sample_index"))
    STORE.report()

    g0, j0 = GEN_TOKENS, JUDGE_META["tokens"]
    smoke = items[:GATE3_ITEMS]
    SWEEP_OUT = []
    for width, expand in SWEEP:
        cid = config_for(width, expand)
        recs = []
        for row in harness.progress(smoke, desc=f"gate 3 beam {width}x{expand}",
                                    unit="item"):
            hit = STORE.get(PRIMARY, cid, row["unique_id"], "0")
            if hit:
                recs.append(hit); continue
            r = beam_search(row, width, expand)
            r.update(run_id=RUN_ID, model_id=PRIMARY, config_id=cid, sample_index="0",
                     judge=JUDGE, judge_kind=JUDGE_KIND,
                     judge_quant=PRM_QUANT or "bf16")
            STORE.append(r); recs.append(r)
        SWEEP_OUT.append(dict(
            width=width, expand=expand, config_id=cid, n=len(recs),
            accuracy=mean([r["correct"] for r in recs]),
            coverage=mean([r.get("beam_coverage", r["correct"]) for r in recs]),
            mean_tokens=mean([r["tokens_generated"] for r in recs]),
            mean_expansions=mean([r["expansions"] for r in recs]),
            forced_rate=mean([r["forced"] for r in recs])))

    ctrl = SWEEP_OUT[0]
    harness.table(
        f"gate 3: the sweep on {len(smoke)} items",
        ["beam", "accuracy", "beam coverage", "left on the table", "tokens/line",
         "judge calls"],
        [[f"{o['width']}x{o['expand']}" + ("  (control, no selection)"
                                           if (o["width"], o["expand"]) == (1, 1) else ""),
          f"{o['accuracy']:.0%}", f"{o['coverage']:.0%}",
          f"{o['coverage'] - o['accuracy']:+.0%}",
          f"{o['mean_tokens']:.0f}", f"{o['mean_expansions']:.0f}"]
         for o in SWEEP_OUT],
        note="coverage is whether ANY line the beam finished was right; accuracy is "
             "whether the judge picked one of them. The gap between them is a selection "
             "failure, not a search failure")

    print(f"\ncontrol (1x1, no selection): {ctrl['accuracy']:.0%}. Page 7's one sampled "
          f"chain on this band: {RUNGS['vote'].get('single_chain', 0.47):.0%}.")
    print("  -> the chopping machinery is "
          + ("harmless; the numbers above are about search"
             if ctrl["accuracy"] >= 0.35 else
             "SUSPECT: the control is far below a plain sampled chain, so read nothing "
             "above it as a result about search"))
    widths = [o["accuracy"] for o in SWEEP_OUT]
    if len(widths) > 2 and widths[-1] < widths[0]:
        print("  -> accuracy FALLS as width rises: the over-optimisation signature "
              "(Stroebl et al.), measured rather than argued")

    best = max(SWEEP_OUT, key=lambda o: o["accuracy"])
    recs = [r for r in read_traces(TRACES_PATH)
            if r["config_id"] == config_for(BEAM_WIDTH, EXPAND)]
    acc = mean([r["correct"] for r in recs]) if recs else float("nan")
    GATE3 = dict(skipped=False, n=len(recs), accuracy=acc, floor=GATE3_MIN,
                 sweep=SWEEP_OUT, control=ctrl, best=best,
                 passed=bool(acc >= GATE3_MIN),
                 mean_tokens=mean([r["tokens_generated"] for r in recs]),
                 mean_expansions=mean([r["expansions"] for r in recs]),
                 forced_rate=mean([r["forced"] for r in recs]),
                 gen_tokens=GEN_TOKENS - g0, judge_tokens=JUDGE_META["tokens"] - j0,
                 judge_kind=JUDGE_KIND, judge_id=JUDGE)
    print(f"\nbeam on {len(recs)} items")
    print(f"  accuracy        {acc:.0%}   (pre-registered floor {GATE3_MIN:.0%})")
    print(f"  tokens/line     {GATE3['mean_tokens']:.0f}   "
          f"expansions/item {GATE3['mean_expansions']:.0f}")
    print(f"  forced to commit {GATE3['forced_rate']:.0%}")
    print(f"  generator tokens {GATE3['gen_tokens']:,}   judge tokens "
          f"{GATE3['judge_tokens']:,}")
    print(f"  projected full arm: ~{GATE3['gen_tokens'] * HARD_N / max(len(recs),1) / 1e6:.1f}M "
          "generator tokens")
    print(f"gate 3: {'PASSED' if GATE3['passed'] else 'FAILED -- report the negative, do not spend the run'}")
else:
    print(f"RUN_MODE={RUN_MODE}: gate 3 needs a GPU and is skipped")

GATES["gate3"] = GATE3

## 7. The arm

In [ ]:
if RUN_MODE == "FULL":
    assert GATE3.get("passed"), "gate 3 did not pass; read its numbers before spending the run"
    for row in harness.progress(items, desc="beam", unit="item"):
        if STORE.get(PRIMARY, CONFIG_ID, row["unique_id"], "0"):
            continue
        r = beam_search(row, BEAM_WIDTH, EXPAND)
        r.update(run_id=RUN_ID, model_id=PRIMARY, config_id=CONFIG_ID, sample_index="0",
                 judge=JUDGE, judge_kind=JUDGE_KIND, judge_quant=PRM_QUANT or "bf16",
                 beam_width=BEAM_WIDTH, expand=EXPAND)
        STORE.append(r)
    print("arm complete")
else:
    print(f"RUN_MODE={RUN_MODE}: the arm is skipped")

## 8. The ladder

Three rungs, one generator, one item set, one grader, one judge. The only thing that
moves is when the judge is allowed to look.

In [ ]:
DERIVED = None

if TRACES_PATH.exists() and RUNGS.get("greedy"):
    all_beams = read_traces(TRACES_PATH)
    by_cfg = {}
    for r in all_beams:
        by_cfg.setdefault(r["config_id"], []).append(r)
    # Records written before the judge kind was part of the id carry only the judge name.
    kind_of = lambda rs: rs[0].get("judge_kind",
                                   "process" if "PRM-7B" in rs[0].get("judge", "") else
                                   "outcome")
    ours = by_cfg.get(CONFIG_ID, [])
    acc = mean([r["correct"] for r in ours]) if ours else float("nan")
    gen_tok_per_item = mean([r["tokens_generated"] for r in ours]) if ours else float("nan")
    g = RUNGS["greedy"]

    ladder = [
        dict(rung="greedy", label=g["label"], accuracy=g["accuracy"],
             judge_calls=0, source="page 6"),
        dict(rung="vote", label=RUNGS["vote"]["label"],
             accuracy=RUNGS["vote"]["accuracy"], judge_calls=0, source="page 7"),
        dict(rung="best_of_n", label=RUNGS["best_of_n"]["label"],
             accuracy=RUNGS["best_of_n"]["accuracy"],
             judge_calls=RUNGS["best_of_n"]["judge_calls"], source="page 8"),
    ]
    for cfg, rs in sorted(by_cfg.items()):
        k = kind_of(rs)
        ladder.append(dict(
            rung=f"beam_{k}", config_id=cfg,
            label=f"beam {rs[0].get('beam_width', BEAM_WIDTH)}x"
                  f"{rs[0].get('expand', EXPAND)}, {k} judge during",
            accuracy=mean([r["correct"] for r in rs]),
            judge_calls=mean([r["expansions"] for r in rs]),
            judge=rs[0].get("judge"), source="this notebook", n_items=len(rs)))
    harness.table(
        f"the ladder on {HARD_N} items, {PRIMARY.split('/')[-1]}"
        + (f", this run's judge = {JUDGE.split('/')[-1]} ({PRM_QUANT or 'bf16'})"
           if ours else ", this run's beam has not been generated yet"),
        ["when the judge looks", "accuracy", "judge calls / item", "items"],
        [[r["label"], f"{r['accuracy']:.0%}", f"{r['judge_calls']:.0f}",
          str(r.get("n_items", HARD_N))] for r in ladder],
        note="every rung uses the same generator, items and grader; gate 2 verified each "
             "judge against the run that established it (exact for the outcome RM, AUC "
             "against page 9's labelled steps for the process PRM)")

    DERIVED = dict(
        ladder=ladder, coverage=RUNGS["vote"].get("coverage"),
        beam=dict(accuracy=acc, n_items=len(ours), width=BEAM_WIDTH, expand=EXPAND,
                  max_depth=MAX_DEPTH, total_cap=TOTAL_CAP, step_stop=STEP_STOP,
                  temperature=T_STEP,
                  mean_tokens=gen_tok_per_item,
                  mean_expansions=mean([r["expansions"] for r in ours]),
                  forced_rate=mean([r["forced"] for r in ours]),
                  truncated_rate=mean([r["truncated"] for r in ours]),
                  mean_depth=mean([r["depth"] for r in ours]),
                  flops_generator=FLOPS(PARAMS, gen_tok_per_item)),
        runnable=dict(ran=["greedy", "beam"],
                      could_not=["breadth-first", "depth-first", "MCTS"],
                      why="an unbounded frontier, no terminal test, and rollouts-to-"
                          "terminal per node respectively, on free-text reasoning steps"),
        item_set_sha256=ITEM_HASH, generator=PRIMARY,
        judge=dict(id=JUDGE, kind=JUDGE_KIND, quant=PRM_QUANT or "bf16",
                   value_at=VALUE_AT, gate2=GATES.get("gate2")))
    if DERIVED["coverage"] and ours:
        print(f"\ncoverage@16 on these items is {DERIVED['coverage']:.0%}: what the pile "
              f"held. This beam returned {acc:.0%}.")
    elif not ours:
        print(f"\nno traces for {CONFIG_ID} yet; the rows above are what is on disk")
else:
    print("no beam traces on disk; nothing to assemble")

## 9. Artifacts

In [ ]:
if DERIVED:
    manifest = dict(
        page=10, run_mode=RUN_MODE,
        run_id=RUN_ID if RUN_MODE in ("QUICK", "FULL") else None,
        generated_at=datetime.now(timezone.utc).isoformat(timespec="seconds"),
        notebook_git_sha=harness.git_sha(), harness=harness.fingerprint(),
        gates={k: v for k, v in GATES.items()},
        items=dict(dataset="math500_l45", n=len(items), seed=SEED,
                   levels=list(HARD_LEVELS), item_set_sha256=ITEM_HASH),
        generator=dict(id=PRIMARY, params=PARAMS, dtype=str(DTYPE), quant="bf16"),
        judge=dict(id=JUDGE, kind=JUDGE_KIND, quant=PRM_QUANT or "bf16",
                   value_at=VALUE_AT, step_sep=STEP_SEP,
                   min_step_chars=MIN_STEP_CHARS,
                   reading=("one probability per step, read at the LAST step of a partial "
                            "line" if JUDGE_KIND == "process" else
                            "a value pooled at the last token of a partial line"),
                   rebuilt_on="transformers Qwen2Model; both checkpoints ship modelling "
                              "code that is broken under transformers 5.x (NaN for "
                              "Skywork, finite-but-wrong for the Math PRM)"),
        beam=dict(width=BEAM_WIDTH, expand=EXPAND, max_depth=MAX_DEPTH,
                  step_max_tokens=STEP_MAX_TOKENS, total_cap=TOTAL_CAP,
                  step_stop=STEP_STOP, temperature=T_STEP, top_p=TOP_P,
                  answer_allowance=ANSWER_ALLOWANCE, force_stem=FORCE_STEM),
        sources=["out/page_06", "out/page_07", "out/page_08"])
    DERIVED_PATH.write_text(json.dumps(DERIVED, indent=2))
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
    print(f"wrote {DERIVED_PATH} ({harness.kb(DERIVED_PATH)})")
    print(f"wrote {MANIFEST_PATH} ({harness.kb(MANIFEST_PATH)})")
else:
    print("nothing derived; no artifacts written")

print("\ngates:", {k: (v.get("passed") if "passed" in v else v.get("reproduced"))
                   for k, v in GATES.items()})

## Notes for re-derivation

- **The judge is held fixed and only its timing moves.** Gate 2 requires this notebook's
  rebuilt Skywork PRM to reproduce page 8's own shipped scores to 1e-3. Without that the
  ladder compares two judges rather than two timings, and the arm means nothing.
- **The generator, items, grader and line budget are pages 6 to 8's.** The beam gets the
  same 2,048 tokens per line that a single chain got, so "search" is not smuggling in a
  longer budget.
- **The step boundary is a claim.** `STEP_STOP` is a blank line, page 9's first cascade
  rule, and it is the convention this judge's own released usage splits on. Cutting steps
  somewhere else would score the model at positions it was never calibrated for.
- **Three of section 2's five strategies could not run**, and the page says which and why
  rather than quietly showing two. That is a limit of our hardware and of free-text
  steps, not a result about the algorithms.
- **What this notebook may not conclude.** A beam that fails here has not shown that
  search fails; it has shown that *this* beam, with *this* judge, at *this* width, did
  not beat the rung below on 100 items. Wu et al., cited on page 11, measured a stronger
  search on a different family and found a crossing; nothing here contradicts that.